# Transformer (synthetic image generator)

In [1]:
import torch
import torch.nn as nn

In [ ]:
class EchocardiogramTransformer(nn.Module):
    def __init__(self, vocab_size=1024, d_model=512, nhead=8, num_layers=6):
        super().__init__()
        # 1. El diccionario de 1024 palabras que mencionas
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # 2. Positional Encoding (Capa para dar noción de orden)
        self.pos_encoder = nn.Parameter(torch.zeros(1, 5000, d_model)) 
        
        # 3. El corazón del modelo: El Transformer Decoder
        # Usamos Decoder porque predice la secuencia N+1 basada en N
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        
        # 4. Capa final para decidir cuál es el siguiente índice del codebook
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x: [Batch, Sequence_Length] (lista de índices/tokens)
        
        # Convertimos índices a vectores continuos usando el codebook
        x = self.embedding(x) # [Batch, Seq, d_model]
        
        # Sumamos la posición (importante para mantener geometría y movimiento)
        x = x + self.pos_encoder[:, :x.size(1), :]
        
        # Máscara Causal: Evita que el modelo "haga trampa" viendo el futuro
        mask = self._generate_square_subsequent_mask(x.size(1)).to(x.device)
        
        # Procesamiento de atención
        output = self.transformer_decoder(tgt=x, memory=x, tgt_mask=mask)
        
        # Proyectamos al tamaño del vocabulario (1024)
        logits = self.output_layer(output)
        return logits

    def _generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask